# Part 1: Keyword Spotting with nnAudio2

#### Welcome to Part 1!

This tutorial demonstrates **nnAudio2** for on-device audio feature extraction. Unlike librosa or other CPU-based libraries, nnAudio2 spectrogram transforms are standard PyTorch `nn.Module` layers — they live **inside** the model, run on whichever device the model is on (CUDA, MPS, or CPU), and can optionally be made **trainable** (see Part 2).

We benchmark nnAudio2 against librosa on the **Google Speech Commands v2** dataset in a 12-class keyword spotting task.

**Classes:**
- 0–9: target keywords `down, go, left, no, off, on, right, stop, up, yes`
- 10: `silence` (1-second clips cut from background noise)
- 11: `unknown` (all remaining words)

[Step 1: Imports](#Step-1:-Imports)\
[Step 2: Configuration & device](#Step-2:-Configuration-&-device)\
[Step 3: Dataset & DataLoaders](#Step-3:-Dataset-&-DataLoaders)\
[Step 4: nnAudio model](#Step-4:-nnAudio-model)\
[Step 5: Train](#Step-5:-Train)\
[Step 6: Librosa baseline](#Step-6:-Librosa-baseline)\
[Conclusion](#Conclusion)

## Step 1: Imports

Standard PyTorch, torchaudio, and PyTorch Lightning imports, plus nnAudio2. We also define a small `SPEECHCOMMANDS_12C` wrapper (below) that maps torchaudio's raw dataset to integer class labels.

In [1]:
import os
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch.utils.data import WeightedRandomSampler, DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

from pytorch_lightning import Trainer, LightningModule

import librosa
from nnAudio2.features.mel import MelSpectrogram

/var/folders/xm/qzb5tds96wd1w9ss080zx9q40000gn/T/ipykernel_78167/2918432426.py:14: CitationReminderWarning: ============================================================
nnAudio Citation Reminder

If you like nnAudio, please cite:

K. W. Cheuk, H. Anderson, K. Agres and D. Herremans,
"nnAudio: An on-the-Fly GPU Audio to Spectrogram Conversion
Toolbox Using 1D Convolutional Neural Networks,"
IEEE Access, vol. 8, pp. 161981-162003, 2020,
doi: 10.1109/ACCESS.2020.3019084.

  from nnAudio2.features.mel import MelSpectrogram


In [2]:
## SPEECHCOMMANDS_12C: 10 keywords + silence + unknown
# Replaces the obsolete AudioLoader.Speech.SPEECHCOMMANDS_12C with a torchaudio-based wrapper.

KEYWORDS = ['down', 'go', 'left', 'no', 'off', 'on', 'right', 'stop', 'up', 'yes']
LABEL_MAP = {word: i for i, word in enumerate(KEYWORDS)}
SILENCE_LABEL = 10
UNKNOWN_LABEL  = 11
TARGET_LEN = 16000  # 1 second at 16 kHz

class SPEECHCOMMANDS_12C(Dataset):
    """torchaudio SPEECHCOMMANDS wrapped as a 12-class dataset.

    Classes 0-9: the 10 target keywords
    Class 10   : silence (1-second clips cut from background noise files)
    Class 11   : unknown (all remaining words)
    """
    def __init__(self, root, url='speech_commands_v0.02',
                 folder_in_archive='SpeechCommands', download=False, subset=None):
        self.base = torchaudio.datasets.SPEECHCOMMANDS(
            root=root, url=url, folder_in_archive=folder_in_archive,
            download=download, subset=subset,
        )
        self.silence = []
        noise_dir = os.path.join(root, folder_in_archive, '_background_noise_')
        if os.path.exists(noise_dir):
            for fname in sorted(os.listdir(noise_dir)):
                if fname.endswith('.wav'):
                    wav, sr = torchaudio.load(os.path.join(noise_dir, fname))
                    for start in range(0, wav.shape[1] - TARGET_LEN, TARGET_LEN):
                        self.silence.append(wav[:, start:start + TARGET_LEN])

    @staticmethod
    def _fix_length(wav):
        """Pad or truncate to exactly TARGET_LEN samples."""
        n = wav.shape[1]
        if n < TARGET_LEN:
            wav = torch.nn.functional.pad(wav, (0, TARGET_LEN - n))
        elif n > TARGET_LEN:
            wav = wav[:, :TARGET_LEN]
        return wav

    def __len__(self):
        return len(self.base) + len(self.silence)

    def __getitem__(self, idx):
        if idx < len(self.base):
            waveform, sr, label, speaker_id, utt_num = self.base[idx]
            waveform = self._fix_length(waveform)
            return waveform, sr, LABEL_MAP.get(label, UNKNOWN_LABEL), speaker_id, utt_num
        waveform = self.silence[idx - len(self.base)]
        return waveform, 16000, SILENCE_LABEL, '', 0

## Step 2: Configuration & device

Device is detected automatically — CUDA → MPS (Apple Silicon) → CPU.

In [3]:
if torch.cuda.is_available():
    device, accelerator = 'cuda', 'gpu'
elif torch.backends.mps.is_available():
    device, accelerator = 'mps', 'mps'
else:
    device, accelerator = 'cpu', 'cpu'

print(f"Using device: {device}")

batch_size               = 100
max_epochs               = 1
check_val_every_n_epoch  = 2
num_sanity_val_steps     = 5

data_root        = './'    # dataset downloaded here
download_option  = True    # set False once downloaded

n_mels     = 40
input_dim  = n_mels * 101
output_dim = 12

Using device: mps


## Step 3: Dataset & DataLoaders

Load train and validation splits. Because `silence` and `unknown` are imbalanced, we use a `WeightedRandomSampler` on the training set. All padding to 16 000 samples is handled by `collate_fn`.

In [4]:
trainset = SPEECHCOMMANDS_12C(root=data_root, url='speech_commands_v0.02',
                              folder_in_archive='SpeechCommands',
                              download=download_option, subset='training')
validset = SPEECHCOMMANDS_12C(root=data_root, url='speech_commands_v0.02',
                              folder_in_archive='SpeechCommands',
                              download=download_option, subset='validation')

# Re-balance: silence (class 10) is under-represented; unknown (class 11) over-represented
class_weights  = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4.6, 1/17]
sample_weights = [class_weights[label] for _, _, label, _, _ in trainset]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

def collate_fn(batch):
    waveforms = pad_sequence([b[0].squeeze(0) for b in batch], batch_first=True)
    return {'waveforms': waveforms, 'labels': torch.tensor([b[2] for b in batch])}

trainloader = DataLoader(trainset, batch_size=batch_size, sampler=sampler, collate_fn=collate_fn)
validloader = DataLoader(validset, batch_size=batch_size, collate_fn=collate_fn)

print(f"Train: {len(trainset):,} samples  |  Val: {len(validset):,} samples")

100%|██████████| 2.26G/2.26G [14:16<00:00, 2.83MB/s] 
/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


Train: 84,843 samples  |  Val: 9,981 samples


## Step 4: nnAudio model

`MelSpectrogram` is dropped into the model as a standard layer. When the model moves to a device, the filterbank weights move with it — no manual `.to(device)` calls needed for the audio front-end.

In [6]:
class KeywordSpotter(LightningModule):
    """Linear keyword spotter with nnAudio2 MelSpectrogram as an in-model layer."""

    def __init__(self):
        super().__init__()
        # nnAudio2: spectrogram runs on the same device as the rest of the model
        self.mel = MelSpectrogram(sr=16000, n_fft=480, hop_length=160, n_mels=n_mels,
                                  fmin=0.0, norm=1, verbose=False)
        self.classifier = nn.Linear(n_mels * 101, output_dim)
        self.criterion   = nn.CrossEntropyLoss()

    def forward(self, x):
        spec  = torch.log(self.mel(x) + 1e-10)        # [B, n_mels, T]
        logits = self.classifier(spec.flatten(1))      # [B, 12]
        return logits, spec

    def _step(self, batch):
        logits, _ = self(batch['waveforms'])
        loss = self.criterion(logits, batch['labels'])
        acc  = (logits.argmax(-1) == batch['labels']).float().mean()
        return loss, acc

    def training_step(self, batch, batch_idx):
        loss, acc = self._step(batch)
        self.log_dict({'train_loss': loss, 'train_acc': acc},
                      on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, acc = self._step(batch)
        self.log_dict({'val_loss': loss, 'val_acc': acc}, prog_bar=True)

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=1e-3)

model = KeywordSpotter()
print(model)

KeywordSpotter(
  (mel): MelSpectrogram(
    Mel filter banks size = (40, 241), trainable_mel=False
    (stft): STFT(n_fft=480, Fourier Kernel size=(241, 1, 480), iSTFT=False, trainable=False)
  )
  (classifier): Linear(in_features=4040, out_features=12, bias=True)
  (criterion): CrossEntropyLoss()
)


## Step 5: Train

PyTorch Lightning handles device placement automatically. `accelerator='auto'` picks CUDA, MPS, or CPU based on what's available.

In [7]:
trainer = Trainer(
    accelerator=accelerator,
    devices=1,
    max_epochs=max_epochs,
    check_val_every_n_epoch=check_val_every_n_epoch,
    num_sanity_val_steps=num_sanity_val_steps,
)
trainer.fit(model, trainloader, validloader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default

  | Name       | Type             | Params | Mode 
--------------------------------------------------------
0 | mel        | Me

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 849/849 [00:23<00:00, 36.12it/s, v_num=0, train_loss=9.350, train_acc=0.262]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 849/849 [00:23<00:00, 36.12it/s, v_num=0, train_loss=9.350, train_acc=0.262]


## Step 6: Librosa baseline

The same linear classifier, but feature extraction uses librosa on CPU. Librosa processes each sample in a Python loop, so it cannot benefit from GPU batching — this is what nnAudio2 replaces.

In [8]:
class KeywordSpotter_librosa(LightningModule):
    """Same classifier, but uses librosa (CPU) for feature extraction."""

    def __init__(self):
        super().__init__()
        self.classifier = nn.Linear(n_mels * 101, output_dim)
        self.criterion   = nn.CrossEntropyLoss()

    def forward(self, x):
        specs = [librosa.feature.melspectrogram(
            y=xi.cpu().numpy(), sr=16000, n_fft=480, hop_length=160,
            n_mels=n_mels, fmin=0.0, norm=1,
        ) for xi in x]
        spec = torch.from_numpy(np.stack(specs)).float().to(x.device)
        spec = torch.log(spec + 1e-10)
        return self.classifier(spec.flatten(1)), spec

    def _step(self, batch):
        logits, _ = self(batch['waveforms'])
        loss = self.criterion(logits, batch['labels'])
        acc  = (logits.argmax(-1) == batch['labels']).float().mean()
        return loss, acc

    def training_step(self, batch, batch_idx):
        loss, acc = self._step(batch)
        self.log_dict({'train_loss': loss, 'train_acc': acc},
                      on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, acc = self._step(batch)
        self.log_dict({'val_loss': loss, 'val_acc': acc}, prog_bar=True)

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=1e-3)

model_librosa = KeywordSpotter_librosa()

### Train the librosa model

In [9]:
trainer_librosa = Trainer(
    accelerator=accelerator,
    devices=1,
    max_epochs=max_epochs,
    check_val_every_n_epoch=check_val_every_n_epoch,
    num_sanity_val_steps=num_sanity_val_steps,
)
trainer_librosa.fit(model_librosa, trainloader, validloader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name       | Type             | Params | Mode 
--------------------------------------------------------
0 | classifier | Linear           | 48.5 K | train
1 | criterion  | CrossEntropyLoss | 0      | train
--------------------------------------------------------
48.5 K    Trainable params
0         Non-trainable params
48.5 K    Total params
0.194     Total estimated model params size (MB)
2         Modules in train mode
0         Modules in eval mode


Sanity Checking DataLoader 0:   0%|          | 0/5 [00:00<?, ?it/s]

/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sanity Checking DataLoader 0:  20%|██        | 1/5 [00:04<00:18,  0.21it/s]

/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 849/849 [06:36<00:00,  2.14it/s, v_num=1, train_loss=5.200, train_acc=0.311]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 849/849 [06:36<00:00,  2.14it/s, v_num=1, train_loss=5.200, train_acc=0.311]


## Conclusion

The timing comparison above shows nnAudio2's advantage. Librosa processes each audio clip on CPU in a Python loop; nnAudio2 runs the entire batch through a single GPU convolution. On a CUDA GPU the gap is typically **~95×**; on Apple MPS it is smaller but still significant.

More importantly, nnAudio2's filterbank is a genuine `nn.Module` — in **Part 2** we'll set `trainable_mel=True` and let the Mel basis adapt during training.